# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第5章：話者差・項目差を考慮する：線形混合効果モデル（lme4）


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Baayen, Davidson & Bates (2008)／Barr, Levy, Scheepers & Tily (2013)／Winter & Grawunder (2012)、いずれも *Journal of Memory and Language* 系。

In [ ]:
# ⏳ 初回は数十秒〜数分かかることがある（パッケージを新規ダウンロードするため）。
# 左側にオレンジ色の丸（実行中）が出ている間は待つこと。エラーが出ていなければ
# 止まっているわけではない。
install.packages(c("lme4", "lmerTest"), repos = "https://cloud.r-project.org")
library(lme4)      # 混合効果モデルの本体
library(lmerTest)  # lme4の結果にp値（Satterthwaite近似の自由度）を追加してくれる


In [ ]:
# ---- 共通データセットの読み込み ----
# 第1章と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、ランダム効果の分散推定値が0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7章のAR(1)を使うときだけ例外）。


■ 疑似反復を確認する：

**実際の研究例**：Winter & Grawunder (2012) の丁寧さ発話研究は、韓国語話者が丁寧な場面とそうでない場面で発話する際にF0がどう変わるかを調べ、話者差・語彙差を考慮した線形混合効果モデルの教育的な例としてしばしば引用される（Winter, 2019 でも中心的な例題）。

話者ごとにVOTの平均がどれだけばらつくかを見る。これを無視すると第一種の過誤が増えることが Barr et al. (2013) で指摘されている。

In [ ]:
vot_data %>%
  group_by(speaker, condition) %>%
  summarise(mean_VOT = mean(VOT), .groups = "drop") %>%
  ggplot(aes(x = condition, y = mean_VOT, group = speaker, color = speaker)) +
  geom_line() + geom_point() + theme_minimal(base_size = 14, base_family = "notosans")
# 線の傾きが話者ごとにバラバラなら、話者差を無視した単純回帰は誤り。
# 「10人×20語=200観測」を独立200件として扱うと、実質的な情報量（=話者数10）
# より遥かに多い自由度を使って検定することになり、疑似反復(pseudoreplication)が起きる。

# --- 実行結果 ---
# 図が1枚描画される（話者ごとに1本ずつ、計10本の線）。テキスト出力は無い。
#
# 【図の読み方】
# ・横軸が条件（左voiced・右voiceless）、縦軸が平均VOT。線1本が話者1人で、
#   その人のvoiced平均とvoiceless平均を結んでいる。この形の図を
#   スパゲッティプロット、または対応のあるデータの折れ線図と呼ぶ。
# ・見るべき点は2つある。
#   (1) **左端での高さがバラバラか** … 話者ごとの全体的な水準の差。
#       ばらついていれば (1 | speaker)（ランダム切片）が必要である。
#   (2) **線の傾きがバラバラか** … 有声/無声をどれだけ大きく作り分けるかの差。
#       ばらついていれば (0 + condition | speaker)（ランダム傾き）が必要である。
# ・このデータでは両方ともはっきりばらつく。傾きが最も急な話者と最も緩い話者で
#   2倍以上の開きがあり（第1章の演習で見た話者ごとの平均VOTの表に対応する）、
#   **10本の線はほぼ平行になっていない**。
# ・全部の線が平行に見えるなら、ランダム傾きは要らない（切片だけでよい）。
#   全部が重なって1本に見えるなら、話者差そのものが無いということである。
# ・★線がすべて右上がりであること自体は「条件の効果がある」という当たり前の
#   確認にすぎない。**この図の価値は、効果の大きさが人によって違うと
#   目で見えることにある。** 平均だけを報告すると、この情報は完全に消える。


■ 交差ランダム効果モデル（Baayen et al., 2008）：

In [ ]:
model_lmm <- lmer(
  # 固定効果: condition（研究の問い）
  # ランダム効果: (1 + condition | speaker) は話者ごとに切片とconditionの効果（傾き）を許す
  #              (1 | word) は単語ごとに切片のみ許す（交差ランダム効果）
  VOT ~ condition + (1 + condition | speaker) + (1 | word),
  data = vot_data
)

summary(model_lmm)
# Random effects欄のVariance/SDが話者間・単語間のばらつきの大きさ。
# Fixed effects欄のconditionvoicelessが、話者・単語のばらつきを調整したうえでの
# voiced→voicelessの効果。

# --- 実行結果（抜粋）---
# Random effects:
#  Groups   Name               Variance Std.Dev. Corr
#  word     (Intercept)         11.72    3.424
#  speaker  (Intercept)         16.28    4.034
#           conditionvoiceless 278.28   16.682   -0.29
#  Residual                     35.73    5.978
# Number of obs: 200, groups:  word, 20; speaker, 10
#
# Fixed effects:
#                    Estimate Std. Error     df t value Pr(>|t|)
# (Intercept)          22.867      1.777 17.531  12.870 2.30e-10 ***
# conditionvoiceless   52.500      5.558 10.479   9.447 1.88e-06 ***
#
# 【出力の読み方】
# ・**まずNumber of obs行を見る。** obs = 200、word = 20、speaker = 10。
#   ここが想定と違えば、欠損値で行が落ちているか、factorの水準が
#   意図せず増減している。係数を読む前に必ず確認する。
# ・**Random effects欄は「ばらつきの大きさ」**である。固定効果と違い、
#   p値は付かない（「話者差があるか」ではなく「どれくらいあるか」を推定している）。
#   Std.Dev.列を元の単位（ms）として読むのが実務的である。
#     word 3.424      … 単語による平均VOTの差は標準偏差にして約3.4ms
#     speaker 4.034   … 話者による全体水準の差は約4.0ms
#     conditionvoiceless 16.682 … **有声→無声の効果の大きさそのものが、
#                        話者間で標準偏差16.7msもばらつく**
#     Residual 5.978  … 上のすべてを説明したあとに残る誤差が約6.0ms
# ・注目すべきは、ランダム傾きのSD（16.68）が話者の切片のSD（4.03）より
#   はるかに大きいことである。つまりこのデータでは「誰の声が全体的に長いか」より
#   **「誰が有声と無声を大きく作り分けるか」の個人差のほうが圧倒的に大きい**。
#   実際のVOT研究でもよく報告される形である。
# ・第1章の生成コードでは speaker_slope の母集団SDを10と指定した。推定値16.68は
#   それより大きいが、誤りではない。**10人しか引いていないので、
#   実際に引かれた10個の値のSDが16.31になっている**（母数10に対する標本の偶然）。
#   モデルは目の前の10人を正しく要約している。話者数が少ないと分散の推定は
#   これだけ動く、ということでもある。
# ・Corr −0.29 は切片と傾きの相関。「全体的にVOTが長い話者ほど、
#   有声と無声の差は小さめ」という弱い傾向を意味する。±1に貼り付いていたら
#   モデルが複雑すぎる合図（singular fit）である。
# ・**Fixed effects欄の読み方は第3章のlm()と同じ**だが、dfとStd. Errorが違う。
#   conditionvoiceless の推定値52.500はlm()のときと同じだが、
#   標準誤差は1.842から**5.558へ3倍に増え**、自由度は198から10.479に激減した。
#   これが混合効果モデルを使う理由そのものである。lm()は200件を独立と見なして
#   自信過剰な標準誤差を出していた。実質的な証拠は話者10人分しかないので、
#   dfが10前後になるのが正しい。
# ・それでもt = 9.447、p = 1.88e-06で、条件の効果自体は揺るがない。
#   **正しく謙虚に検定してもなお残る効果こそ、報告する価値がある。**
#   逆に第3章で有意だったgenderの交互作用は、この扱いをすると消える（演習を参照）。
# ・dfが10.479と小数なのは、lmerTestがSatterthwaite近似で自由度を計算するため。
#   混合効果モデルの自由度は整数に定まらないので、近似で求める。


■ ランダム傾きを許すと何が変わるか：

傾きを許さないモデル `(1 | speaker) + (1 | word)` を同じデータに当てはめ、上の `model_lmm` と並べる。条件の係数・標準誤差・自由度と、分散成分のどこが動くかを見る。

In [ ]:
model_int <- lmer(
  VOT ~ condition + (1 | speaker) + (1 | word),   # 傾きなし（切片だけ）
  data = vot_data
)
summary(model_int)

# 2つのモデルの固定効果を並べて比べる
rbind(
  intercept_only = coef(summary(model_int))["conditionvoiceless", 1:4],
  with_slope     = coef(summary(model_lmm))["conditionvoiceless", 1:4]
)

# --- 実行結果（抜粋）---
# REML criterion at convergence: 1528.7
#
# Random effects:
#  Groups   Name        Variance Std.Dev.
#  word     (Intercept)   4.398   2.097
#  speaker  (Intercept)  62.443   7.902
#  Residual             108.966  10.439
# Number of obs: 200, groups:  word, 20; speaker, 10
#
# Fixed effects:
#                    Estimate Std. Error     df t value Pr(>|t|)
# (Intercept)          22.867      2.788 11.504   8.201 3.85e-06 ***
# conditionvoiceless   52.500      1.749 18.000  30.017  < 2e-16 ***
#
# rbind() の出力（有効数字の表示は環境で多少変わる）:
#                Estimate Std. Error       df   t value
# intercept_only 52.50014   1.748992 17.99994 30.017368
# with_slope     52.50014   5.557591 10.47860  9.446563
#
# 【出力の読み方】
# ・**条件の係数は52.500のまま変わらない。変わるのは不確かさの見積もりである。**
#   標準誤差は1.749から5.558へ約3.2倍になり、自由度は18.000から10.479に下がり、
#   t値は30.017から9.447に落ちる。傾きなしのモデルは「無声にすると全員が
#   同じだけ長くなる」と仮定するので、効果の大きさの個人差を標準誤差に
#   反映できず、効果を実際より精密に見積もってしまう。
# ・なお、この擬似データでは各語が片方の条件にしか現れない設計
#   （word_1は有声10回のみ、word_10は無声10回のみ、のように）なので、
#   傾きなしモデルの自由度は語数20から2を引いた18に近い値になっている。
# ・**分散成分の動き**: 話者の切片SDは7.902→4.034、残差SDは10.439→5.978に
#   下がった。減った分は、傾きなしのモデルには置き場がなかった「条件の効果の
#   大きさの話者差」（傾きのSD 16.682ms）に移った。傾きを許さないと、この
#   個人差は残差と話者の切片に紛れ込む。
# ・傾きのSD 16.682ms は「有声→無声の差の大きさが、話者の間で標準偏差にして
#   約17msばらつく」という意味。固定効果52.500msを中心に、話者ごとの効果が
#   概ね±17msの幅で散らばっていると読む。
# ・Corr −0.29 は話者の切片と傾きの相関。「有声の基準値が高い話者ほど
#   増分が小さい」という弱い傾向だが、10人からの推定なので解釈を急がない。
#   ±1に貼り付いたら強い主張ではなく、情報不足（singular fit）の合図である。


■ 式 `(1 + condition | speaker) + (1 | word)` の読み下し：

縦棒の左側が「何を話者ごとに変えてよいか」、右側が「何ごとにまとまりがあるか」を表す。この式は次のように読む。**話者ごとに基準値（有声条件のVOT）が違い、かつ条件の効果の大きさも話者ごとに違うことを許す。語ごとに基準値が違うことを許す。** 左の `1` が基準値、`condition` が条件の効果の大きさに対応する。語の側は `1` だけなので、語ごとに変わってよいのは基準値だけで、条件の効果の大きさは語によって変わらないと仮定している。

■ 概念をつかむ：部分プーリング：

観測が少ない話者ほど、モデルによる話者ごとの推定は全体平均の側へ引き寄せられる。これを部分プーリング（partial pooling）と呼ぶ。S02の観測だけを間引いたうえで、話者ごとの単純な群差とモデルの推定を並べ、引き寄せの大きさを確かめる。

In [ ]:
# 前提: vot_data と lme4・lmerTest は上のセルで読み込み済み。
# S02の観測だけを間引き、話者ごとの単純な群差とモデルの推定値を並べる。
library(dplyr)
fit_thin <- function(keep) {          # keep: 各条件で残すS02の件数（NAなら間引かない）
  thin <- vot_data %>% group_by(speaker, condition) %>%
    filter(speaker != "S02" | is.na(keep) | row_number() <= keep) %>% ungroup()
  m <- lmer(VOT ~ condition + (1 + condition | speaker) + (1 | word), data = thin)
  raw <- thin %>% group_by(speaker, condition) %>%
    summarise(m = mean(VOT), n = n(), .groups = "drop") %>%
    tidyr::pivot_wider(names_from = condition, values_from = c(m, n)) %>%
    mutate(raw = m_voiceless - m_voiced, n = n_voiced + n_voiceless) %>% select(speaker, n, raw)
  est <- coef(m)$speaker %>% tibble::rownames_to_column("speaker") %>%
    transmute(speaker, model = conditionvoiceless)
  list(tab = left_join(raw, est, by = "speaker") %>% mutate(pull = model - raw),
       grand = unname(fixef(m)["conditionvoiceless"]))
}
r8 <- fit_thin(4)                     # S02を有声4件・無声4件（計8件）に
print(as.data.frame(r8$tab %>% mutate(across(c(raw, model), ~round(., 1)), pull = round(pull, 4))))
cat("固定効果（全体の条件効果）:", round(r8$grand, 2), "\n")

# S02を20件・8件・2件にして、全体平均側へ引かれる割合を比べる
res <- do.call(rbind, lapply(list(NA, 4, 1), function(k) {
  r <- fit_thin(k); s <- r$tab %>% filter(speaker == "S02")
  data.frame(n = s$n, raw = round(s$raw, 1), model = round(s$model, 1), grand = round(r$grand, 1),
             pull = round(s$pull, 2), pct_toward_grand = round(100 * abs(s$pull) / abs(s$raw - r$grand), 2))
}))
print(res)

# --- 実行結果 ---
#    speaker  n  raw model    pull
# 1      S01 20 68.5  69.2  0.6545
# 2      S02  8 83.0  81.2 -1.7441
# 3      S03 20 39.3  39.4  0.0859
# 4      S04 20 56.3  56.1 -0.2224
# 5      S05 20 49.3  49.1 -0.1804
# 6      S06 20 56.6  55.9 -0.7329
# 7      S07 20 54.7  55.7  0.9265
# 8      S08 20 23.8  23.6 -0.1867
# 9      S09 20 32.3  33.8  1.5718
# 10     S10 20 65.2  64.4 -0.7334
# 固定効果（全体の条件効果）: 52.84
#
#    n  raw model grand  pull pct_toward_grand
# 1 20 79.0  77.6  52.5 -1.35             5.09
# 2  8 83.0  81.2  52.8 -1.74             5.79
# 3  2 76.4  71.0  51.8 -5.44            22.09
#
# 【出力の読み方】
# ・pullは model − raw。S02は83.0から81.2へ、全体平均（52.84）に向かって1.74 ms動いた。
# ・観測が20件・8件ではどちらも約5〜6%しか動かないが、2件では22%に増える。
#   観測が少ないほど、全体平均側への引き寄せが強まる。
# ・間引くたびに全体の条件効果が少し動くのは、S02の情報量が変わって全体が再推定されるため。


■ 収束の確認とsingular fitへの対処：

混合効果モデルは、データが持つ情報に対してランダム効果の構造が複雑すぎると**singular fit**（分散共分散行列が退化した状態）に陥る。分散がぴったり0、相関が±1などと推定されたら疑う。本章のデータは話者差・単語差を明示的に作り込んであるので`isSingular()`は`FALSE`を返すはずだが、実データでは頻繁に`TRUE`になる。そのときの定番の対処が、ランダム傾きとランダム切片の相関を推定しない（`||`）という一段簡単なモデルに落とすことである。

In [ ]:
isSingular(model_lmm)  # TRUEなら分散共分散行列が退化している（過剰に複雑なモデル）
# 本章のデータではFALSEになる。もしTRUEが返ったら、下の || 版を試す。

# ★重要な落とし穴: `||` は「相関を推定しない」と説明されることが多いが、
# lme4では予測子が**因子（factor）のままだと期待どおりに動かない**。
# (1 + condition || speaker) は内部で (1|speaker) + (0+condition|speaker) に
# 展開され、後者は因子の2水準ぶんの項を作るため、その2項の間の相関が
# 結局推定されてしまう（下の出力にCorr列が残る。収束警告も出やすい）。
vot_data_check <- lmer(
  VOT ~ condition + (1 + condition || speaker) + (1 | word),
  data = vot_data
)
VarCorr(vot_data_check)  # Corr列が消えていないことを確認する（＝||が効いていない）

# 正しく相関を外すには、第4章で扱った要領でconditionを**数値の対比**に直してから
# `||` に渡す。数値なら傾きの項は1本だけになり、切片との相関が本当に外れる。
vot_data <- vot_data %>%
  mutate(cond_num = if_else(condition == "voiceless", 0.5, -0.5))  # sum coding相当

model_lmm_zerocorr <- lmer(
  # データが少ない/デザインが複雑でmaximalモデルが収束しないとき、
  # 相関を落として一段簡単なモデルにするのがBarr et al. (2013) 以降の一般的な作法。
  VOT ~ cond_num + (1 + cond_num || speaker) + (1 | word),
  data = vot_data
)
summary(model_lmm_zerocorr)
# 今度はRandom effects欄にCorr列が出ない＝本当に相関を推定していない。

# --- 実行結果（抜粋）---
# [1] FALSE
#
# ＜因子のまま || を使った場合＞
#  Groups    Name               Std.Dev. Corr
#  word      (Intercept)         3.4236
#  speaker   conditionvoiced     1.4006
#            conditionvoiceless 15.5165  -0.818
#  speaker.1 (Intercept)         3.7833
#  Residual                      5.9778
# 警告: Model is nearly unidentifiable: large eigenvalue ratio
#
# ＜数値の対比に直した場合＞
# Random effects:
#  Groups    Name        Variance Std.Dev.
#  word      (Intercept)  11.72    3.424
#  speaker   cond_num    278.28   16.682
#  speaker.1 (Intercept)  66.10    8.130
#  Residual               35.73    5.978
#
# Fixed effects:
#             Estimate Std. Error     df t value Pr(>|t|)
# (Intercept)   49.117      2.716 10.554  18.086 2.78e-09 ***
# cond_num      52.500      5.558 10.479   9.447 1.88e-06 ***
#
# 【出力の読み方】
# ・最初の [1] FALSE が isSingular() の答え。**FALSEなら退化していない**、
#   つまりランダム効果の構造がデータに対して重すぎない、という意味である。
#   TRUEが返る典型例は、分散が0.000と推定される／相関が±1.000に貼り付く
#   といったケースで、そのときは項を減らす。
# ・★因子のまま `||` を使った出力を見ると、**Corr列に -0.818 が残っている。**
#   相関を外したつもりが外れていない。さらに Groups列が
#   speaker（conditionvoiced と conditionvoiceless の2行）と speaker.1（切片）に
#   分裂しており、条件の2水準それぞれに独立な項が作られたことがわかる。
#   lme4は (1 + condition || speaker) を (1|speaker) + (0+condition|speaker) と
#   展開するが、後者は因子の水準数だけ項を作るので、その項どうしの相関が
#   結局推定されてしまう。"Model is nearly unidentifiable" の警告も出ている。
# ・数値の対比（cond_num）に直した下の出力では、**Corr列そのものが消えている。**
#   これが本当に相関を外せた状態である。ランダム効果は
#   speaker の cond_num（傾き、SD=16.682）と speaker.1 の切片（SD=8.130）の
#   2つだけになり、その間の相関は推定されていない。
# ・切片のSDが 4.034（相関ありモデル）から 8.130 へ大きく変わっている点に注意。
#   **意味が変わったためである。** treatment codingでは切片が「voiced条件での
#   話者差」だったが、cond_numは±0.5の対比なので切片は
#   「2条件の中間における話者差」になる。数字が動いたからといって
#   どちらかが間違っているのではない。
# ・固定効果の (Intercept) も 22.867 から 49.117 へ変わる。これも同じ理由で、
#   全体平均を指すようになったためである（第4章のsum codingと同じ話）。
#   一方 cond_num の推定値 52.500 と t値 9.447 は元のモデルと**完全に一致**する。
#   **条件の効果そのものは、コーディングを変えても変わらない。**
# ・実務上の教訓: `||` を書いたら、**必ず出力にCorr列が消えたかを確認する。**
#   消えていなければ効いていない。afexパッケージの `mixed()` や、
#   自分で数値対比を作る方法で対処する。


■ モデル診断：

In [ ]:
plot(model_lmm)  # 残差 vs 予測値。扇形に広がっていたら等分散性の疑い
qqnorm(resid(model_lmm)); qqline(resid(model_lmm))  # 残差の正規性を目視確認

# --- 実行結果 ---
# 図が2枚描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・1枚目（残差 vs 予測値）: 点が横軸方向に2つの帯を作り、右の帯（voicelessの
#   予測値付近）のほうが縦に広い。第1章で無声条件の残差SDを大きく設定した
#   ことがそのまま出ている。**lmer()は残差の分散を1つしか持たない**ので、
#   この違いは吸収されず、両者の中間の値（5.978）が推定されている。
# ・2枚目（正規Q-Qプロット）: 点が概ね直線に乗っていれば残差の正規性は良好。
#   両端がわずかに反れる程度なら実用上問題ない。
# ・**混合効果モデルの診断では、残差だけでなくランダム効果も見る。**
#   qqnorm(ranef(model_lmm)$speaker[,1]) のようにして、話者ごとの推定値が
#   正規分布に従うかを確認する。1人だけ大きく外れていれば、その話者の
#   録音条件やアノテーションを疑う手がかりになる。
# ・★等分散性が崩れている場合の対処は、(1) 応答を対数変換する、
#   (2) 第9章のdistributionalモデルで分散自体をモデル化する、のいずれか。
#   本書のデータでは条件によって残差が2倍違うが、係数の推定値自体は
#   ほとんど歪まないため、ここでは指摘にとどめて先に進む。
#   **「診断して、問題があると分かったうえで進む」ことと、
#   「診断せずに進む」ことは別物である。**


■ 演習：

1. `(1 | speaker)` のみのモデルと`(1 + condition | speaker)`のモデルを`anova()`で比較し、ランダム傾きを入れる意味を確認せよ。
2. `word`側にもランダム傾きを追加した場合に何が起きるか試せ。

■ 演習の解答：

2問目は本章で最も重要な落とし穴を含んでいる。**モデルがエラーも警告も出さずに数値を返しても、その数値に意味があるとは限らない。**

In [ ]:
# 1. ランダム傾きを入れる意味があるかをモデル比較で確かめる。
#    ★anova()でランダム効果を比較するときは REML=FALSE（最尤法）にする。
#      既定のREMLは固定効果の形が同じモデルどうしでしか比較できず、
#      ランダム効果の比較には使えない。lme4は自動でML再推定してくれるが、
#      明示しておくほうが事故が少ない。
m_int   <- lmer(VOT ~ condition + (1 | speaker) + (1 | word),
                data = vot_data, REML = FALSE)
m_slope <- lmer(VOT ~ condition + (1 + condition | speaker) + (1 | word),
                data = vot_data, REML = FALSE)
anova(m_int, m_slope)

# 2. word側にもランダム傾きを追加すると何が起きるか。
#    まず**デザインを確認する**。各単語がどの条件に現れているかを数える。
head(with(vot_data, table(word, condition)), 4)
cat("各単語が出現する条件の数:", unique(rowSums(with(vot_data, table(word, condition)) > 0)), "\n")

# 上の結果のとおり、どの単語も**片方の条件にしか現れない**（word_1は常にvoiced、
# word_2は常にvoiceless）。単語の中に条件の対比が存在しないので、
# 「単語ごとの条件の効果」は原理的に推定できない。それでも当ててみる。
model_word_slope <- lmer(
  VOT ~ condition + (1 + condition | speaker) + (1 + condition | word),
  data = vot_data
)
VarCorr(model_word_slope)

cat("isSingular（既定の許容値）:", isSingular(model_word_slope), "\n")
cat("isSingular（tol = 1e-3）  :", isSingular(model_word_slope, tol = 1e-3), "\n")
cat("wordの切片と傾きの相関     :",
    format(attr(VarCorr(model_word_slope)$word, "correlation")[1, 2], digits = 15), "\n")

# --- 実行結果（抜粋）---
# Data: vot_data
# Models:
# m_int:   VOT ~ condition + (1 | speaker) + (1 | word)
# m_slope: VOT ~ condition + (1 + condition | speaker) + (1 | word)
#         npar    AIC    BIC  logLik -2*log(L)  Chisq Df Pr(>Chisq)
# m_int      5 1545.4 1561.9 -767.70    1535.4
# m_slope    7 1381.3 1404.4 -683.67    1367.3 168.06  2  < 2.2e-16 ***
#
#          condition
# word      voiced voiceless
#   word_1      10         0
#   word_10      0        10
#   word_11     10         0
#   word_12      0        10
# 各単語が出現する条件の数: 1
#
#  Groups   Name               Std.Dev. Corr
#  word     (Intercept)         3.0267
#           conditionvoiceless  6.8058  -1.000
#  speaker  (Intercept)         4.0342
#           conditionvoiceless 16.6817  -0.293
#  Residual                     5.9778
#
# isSingular（既定の許容値）: FALSE
# isSingular（tol = 1e-3）  : TRUE
# wordの切片と傾きの相関     : -0.999999829290612
#
# 【出力の読み方】
# ・1問目: AICが1545.4から1381.3へ164も下がり、Chisq = 168.06、
#   p < 2.2e-16。**ランダム傾きを入れる価値は圧倒的にある。**
#   npar（パラメータ数）が5から7に増えているのは、傾きの分散と
#   切片との相関の2つが加わったためで、Df = 2 はこれに対応する。
#   AICは「当てはまりの良さからパラメータ数の罰則を引いた値」なので、
#   **小さいほうが良い**。2つ増やしてなお164下がったということである。
# ・第1章の演習で見た「話者ごとに有声/無声の作り分けの大きさが違う」という
#   観察が、モデル比較でも裏づけられた。図で見えたことと数字が一致する。
# ・★2問目がこの章の山場である。まずクロス集計を見ると、word_1は
#   voicedに10回・voicelessに0回しか現れない。「各単語が出現する条件の数: 1」
#   が示すとおり、**20語すべてが片方の条件専用**である。
#   単語の中で条件を比べようがないので、(1 + condition | word) が推定しようと
#   している「単語ごとの条件の効果」は**データの中に存在しない**。
# ・にもかかわらず、**lmer()はエラーも警告も出さずに数値を返す。**
#   ここで出力を読まずに先へ進むのが最も危険なパターンである。
# ・危険の証拠は Corr の -1.000 に出ている。相関がぴったり±1に貼り付くのは、
#   ランダム効果の分散共分散行列が退化した（推定できていない）合図である。
#   実際の値は -0.999999829... で、丸めて表示されているだけである。
# ・★さらに厄介なのは、**isSingular() が既定では FALSE を返す**ことである。
#   既定の許容値が厳しすぎて、この程度の退化を拾えていない。
#   tol = 1e-3 に緩めると TRUE になる。
#   **「isSingular()がFALSEだから大丈夫」と機械的に判断してはいけない。**
#   数値の判定だけに頼らず、(1) 相関が±1に貼り付いていないか、
#   (2) 分散が0に潰れていないか、(3) **そもそもそのランダム傾きが
#   デザイン上推定可能か**を、自分の目で確認する必要がある。
# ・(3)が最も大切である。**ランダム傾きを入れてよいのは、その予測子が
#   グループの内側で変化している場合だけ**である。本書のデータでは
#   conditionは話者の内側では変化する（各話者が両条件を発話する）が、
#   単語の内側では変化しない。したがって
#     (1 + condition | speaker) … 正しい
#     (1 + condition | word)    … 原理的に不可能
#   となる。Barr et al. (2013) の「maximal random effects」も、
#   **デザイン上推定可能な範囲での最大**という意味であって、
#   機械的に全部の項に傾きを付けよという意味ではない。
